# Notebook 08 - Spectral Agent Prototype

## Goal
Train a baseline classifier from MFCC summary vectors.


## Agenda
- Infer labels from filenames/folders
- Extract MFCC statistics
- Train logistic regression
- Read basic metrics


## Concept and Math

A spectral agent learns distribution differences in frequency-domain patterns.
Start with simple classifiers to establish baseline behavior quickly.


In [ ]:
from pathlib import Path
import numpy as np
import librosa as lb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

DATA_ROOT = Path("../dataset")
audio_files = sorted(DATA_ROOT.rglob("*.flac")) + sorted(DATA_ROOT.rglob("*.wav"))

def infer_label(p):
    s = str(p).lower()
    if any(k in s for k in ["fake", "spoof", "synth", "tts", "vc"]):
        return 1
    if any(k in s for k in ["real", "bona", "bonafide", "human"]):
        return 0
    return None

X, y = [], []
for p in audio_files:
    label = infer_label(p)
    if label is None:
        continue
    w, sr = lb.load(p, sr=16000, mono=True)
    mfcc = lb.feature.mfcc(y=w, sr=sr, n_mfcc=20)
    X.append(np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)]))
    y.append(label)

X, y = np.array(X), np.array(y)
print("usable samples:", len(y))
if len(y) >= 20 and len(np.unique(y)) == 2:
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(Xtr, ytr)
    print("test_accuracy:", model.score(Xte, yte))


## PyTorch Equivalent Snippet
Understand the librosa block first, then map it to this snippet.


In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(nn.Linear(40, 32), nn.ReLU(), nn.Linear(32, 2))
example = torch.randn(8, 40)
print(model(example).shape)


## Review Checklist
- What assumptions are hidden in label inference?
- Why are mean and std over MFCC frames used here?
- How will you validate unseen-generator generalization?
